# Gemma-4 26B Top-50 Model Comparison

Compares Notebook 11 Zahid adapter vs Notebook 14 combined Zahid+WLASL adapter on the same WLASL Top-50 val/test data.

No Hub upload. Results are printed and written locally to `model_comparison_11_vs_14.json`.


In [ ]:
# 1) Install deps
import os
os.environ["UNSLOTH_DISABLE_STATISTICS"] = "1"
os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True,max_split_size_mb:256")

!pip -q install --upgrade pip
!pip -q install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" trl peft accelerate datasets huggingface_hub


In [ ]:
# Config
from pathlib import Path

BASE_MODEL = "unsloth/gemma-4-26B-A4B-it"
MAX_SEQ_LENGTH = 8192

MODEL_REPOS = {
    "notebook11_zahid_phase1": "AlexD281/asl-gemma4-26b-a4b-zahid-pretrain-lora",
    "notebook14_zahid_wlasl_combined": "AlexD281/asl-gemma4-26b-a4b-zahid-wlasl-combined-top50-lora",
}
NUM_FRAMES = 30
IMAGE_SIZE = 448
MAX_IMAGES_PER_SAMPLE = 6
EVAL_MAX_SAMPLES = 200
EVAL_DEBUG_ROWS = 20
WORK_DIR = Path("/content/asl_gemma4_26b_model_comparison")
WORK_DIR.mkdir(parents=True, exist_ok=True)
ZAHID_DATA_DIR = WORK_DIR / "zahid_top50"
WLASL_DATA_DIR = WORK_DIR / "wlasl_top50"
ZAHID_DATA_DIR.mkdir(parents=True, exist_ok=True)
WLASL_DATA_DIR.mkdir(parents=True, exist_ok=True)

ZAHID_ZIP_PATH = Path("/content/drive/MyDrive/asl/phase1_zahid_top50_bundle.zip")
WLASL_ZIP_PATH = Path("/content/drive/MyDrive/asl/video_finetune/top50_bundle.zip")
WLASL_ZIP_FALLBACK = Path("/content/drive/MyDrive/asl/wlasl_top50_30x448_bundle.zip")
TOP50_LABELS_FILE = Path("/content/drive/MyDrive/asl/video_finetune/top50/labels.txt")

print("BASE_MODEL:", BASE_MODEL)
print("MODEL_REPOS:", MODEL_REPOS)
print("ZAHID_ZIP_PATH:", ZAHID_ZIP_PATH)
print("WLASL_ZIP_PATH:", WLASL_ZIP_PATH)
print("MAX_IMAGES_PER_SAMPLE:", MAX_IMAGES_PER_SAMPLE)


In [ ]:
# Mount Drive + load allowlist
import json
import re
from google.colab import drive

drive.mount("/content/drive")

assert TOP50_LABELS_FILE.exists(), f"Missing labels file: {TOP50_LABELS_FILE}"
ALLOWED_LABELS = [x.strip().lower() for x in TOP50_LABELS_FILE.read_text(encoding="utf-8").splitlines() if x.strip()]
assert len(ALLOWED_LABELS) == 50 and len(set(ALLOWED_LABELS)) == 50
ALLOWED_SET = set(ALLOWED_LABELS)
ALLOWED_BY_LEN = sorted(ALLOWED_LABELS, key=len, reverse=True)

OUTPUT_CONTRACT_PROMPT = (
    "Identify the ASL sign shown across these frames. "
    "Return exactly one label from the approved list. "
    "Output only the label text in lowercase, with no extra words, punctuation, or JSON."
)

def norm(text):
    return re.sub(r"\s+", " ", str(text).strip().lower())

def normalize_label_with_reason(raw):
    raw = str(raw).strip()
    candidates = [raw, raw.split("\n")[0].strip()]
    try:
        parsed = json.loads(raw)
        if isinstance(parsed, dict):
            for key in ("translation", "label", "gloss", "answer"):
                if isinstance(parsed.get(key), str):
                    candidates.append(parsed[key])
        elif isinstance(parsed, str):
            candidates.append(parsed)
    except Exception:
        pass

    strip_chars = " .,:;!?\"'`()[]{}"
    candidates = [norm(x.strip(strip_chars)) for x in candidates if str(x).strip()]
    for candidate in candidates:
        if candidate in ALLOWED_SET:
            return candidate, "exact_or_wrapper"

    joined = " || ".join(candidates)
    hits = sorted(set(label for label in ALLOWED_BY_LEN if re.search(rf"(?<![a-z]){re.escape(label)}(?![a-z])", joined)))
    if len(hits) == 1:
        return hits[0], "single_mention"
    if len(hits) > 1:
        return "__invalid__", "ambiguous"
    return "__invalid__", "no_match"

print("Allowlist loaded:", len(ALLOWED_LABELS))


In [ ]:
# Build Zahid + WLASL val/test samples
import json
import zipfile
from pathlib import Path

if not WLASL_ZIP_PATH.exists() and WLASL_ZIP_FALLBACK.exists():
    WLASL_ZIP_PATH = WLASL_ZIP_FALLBACK

for name, zip_path in [("Zahid", ZAHID_ZIP_PATH), ("WLASL", WLASL_ZIP_PATH)]:
    assert zip_path.exists(), f"Missing {name} zip: {zip_path}"

def extract_if_needed(zip_path: Path, extract_root: Path):
    has_manifests = any((path.parent / "val.jsonl").exists() and (path.parent / "test.jsonl").exists() for path in extract_root.rglob("train.jsonl"))
    if not has_manifests:
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(extract_root)
        print("Extracted ->", extract_root)

def find_manifest_dir(extract_root: Path):
    required = ["train.jsonl", "val.jsonl", "test.jsonl"]
    def has_required(path: Path):
        return all((path / name).exists() for name in required)
    candidates = sorted({path.parent for path in extract_root.rglob("train.jsonl") if has_required(path.parent)}, key=lambda path: len(path.parts))
    if has_required(extract_root):
        candidates.insert(0, extract_root)
    if not candidates:
        raise FileNotFoundError(f"Could not find train/val/test manifests under {extract_root}")
    return candidates[0]

def find_frames_root(manifest_dir: Path):
    frames_name = f"frames_{NUM_FRAMES}x{IMAGE_SIZE}"
    for candidate in [manifest_dir / frames_name, manifest_dir / "frames"]:
        if candidate.exists():
            return candidate
    hits = [path for path in manifest_dir.rglob("*") if path.is_dir() and path.name in {frames_name, "frames"}]
    return sorted(hits, key=lambda path: len(path.parts))[0] if hits else manifest_dir

extract_if_needed(ZAHID_ZIP_PATH, ZAHID_DATA_DIR)
extract_if_needed(WLASL_ZIP_PATH, WLASL_DATA_DIR)

ZAHID_ROOT = find_manifest_dir(ZAHID_DATA_DIR)
WLASL_ROOT = find_manifest_dir(WLASL_DATA_DIR)
ZAHID_FRAMES_ROOT = find_frames_root(ZAHID_ROOT)

print("ZAHID_ROOT:", ZAHID_ROOT)
print("ZAHID_FRAMES_ROOT:", ZAHID_FRAMES_ROOT)
print("WLASL_ROOT:", WLASL_ROOT)

def read_jsonl(path: Path):
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]

def evenly_subsample(paths, max_items: int):
    if len(paths) <= max_items:
        return paths
    idxs = [round(i * (len(paths) - 1) / (max_items - 1)) for i in range(max_items)]
    return [paths[i] for i in idxs]

def resolve_frame_paths(row, split_name: str, dataset_root: Path, frames_root: Path | None):
    sample_id = str(row.get("sample_id", "")).strip()

    if isinstance(row.get("frame_paths"), list):
        paths = []
        for raw_path in row["frame_paths"]:
            path = Path(str(raw_path).strip())
            if not path.is_absolute():
                path = dataset_root / path
            if path.exists():
                paths.append(str(path.resolve()))
        if paths:
            return paths

    sample_dir = frames_root / split_name / sample_id
    frames = sorted(sample_dir.glob("*.jpg")) + sorted(sample_dir.glob("*.png"))
    if not frames:
        raise FileNotFoundError(f"No frames found for {split_name}/{sample_id} under {sample_dir}")
    return [str(path.resolve()) for path in frames]

def row_to_sample(row, expected_split, dataset_name: str, dataset_root: Path, frames_root: Path | None):
    sample_id = str(row["sample_id"]).strip()
    gloss = norm(row["gloss"])
    split = norm(row.get("split", expected_split))
    assert split == expected_split, f"split mismatch: {sample_id} {split} != {expected_split}"
    assert gloss in ALLOWED_SET, f"label outside allowlist: {gloss}"

    frame_paths = resolve_frame_paths(row, expected_split, dataset_root, frames_root)
    selected_paths = evenly_subsample(frame_paths, MAX_IMAGES_PER_SAMPLE)

    return {
        "sample_id": f"{dataset_name}:{sample_id}",
        "dataset": dataset_name,
        "label": gloss,
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": OUTPUT_CONTRACT_PROMPT},
                    *[{"type": "image", "image": frame_path} for frame_path in selected_paths],
                ],
            },
            {"role": "assistant", "content": [{"type": "text", "text": gloss}]},
        ],
    }

def build_split(dataset_name, dataset_root, frames_root, split_name):
    rows = read_jsonl(dataset_root / f"{split_name}.jsonl")
    return [row_to_sample(row, split_name, dataset_name, dataset_root, frames_root) for row in rows]

zahid_val_data = build_split("zahid", ZAHID_ROOT, ZAHID_FRAMES_ROOT, "val")
zahid_test_data = build_split("zahid", ZAHID_ROOT, ZAHID_FRAMES_ROOT, "test")
wlasl_val_data = build_split("wlasl", WLASL_ROOT, None, "val")
wlasl_test_data = build_split("wlasl", WLASL_ROOT, None, "test")
combined_val_data = zahid_val_data + wlasl_val_data
combined_test_data = zahid_test_data + wlasl_test_data

print("zahid val/test sizes:", len(zahid_val_data), len(zahid_test_data))
print("wlasl val/test sizes:", len(wlasl_val_data), len(wlasl_test_data))
print("combined val/test sizes:", len(combined_val_data), len(combined_test_data))


In [ ]:
# Evaluate and compare
import gc
import json
import os
from getpass import getpass
from pathlib import Path

import torch
from unsloth import FastVisionModel
from huggingface_hub import login, snapshot_download

hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    login(token=hf_token, add_to_git_credential=False)


def predict(model, tokenizer, sample):
    user_msg = [message for message in sample["messages"] if message["role"] == "user"][0]
    inputs = tokenizer.apply_chat_template(
        [user_msg],
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    )
    if hasattr(inputs, "to"):
        inputs = inputs.to(model.device)
    else:
        inputs = {key: value.to(model.device) if hasattr(value, "to") else value for key, value in inputs.items()}

    prompt_len = int(inputs["input_ids"].shape[-1])
    out = model.generate(**inputs, max_new_tokens=12, do_sample=False)
    raw = tokenizer.decode(out[0][prompt_len:], skip_special_tokens=True).strip()
    pred, reason = normalize_label_with_reason(raw)
    return pred, raw, reason


def score_split(model, tokenizer, split_name, data):
    subset = data[:EVAL_MAX_SAMPLES]
    correct = 0
    invalid = 0
    debug = []
    for index, sample in enumerate(subset):
        pred, raw, reason = predict(model, tokenizer, sample)
        gold = sample["label"]
        correct += int(pred == gold)
        invalid += int(pred == "__invalid__")
        if index < EVAL_DEBUG_ROWS:
            debug.append({
                "sample_id": sample.get("sample_id"),
                "gold": gold,
                "pred": pred,
                "raw_pred": raw,
                "reason": reason,
            })

    total = len(subset)
    return {
        "samples": total,
        "accuracy": correct / max(total, 1),
        "invalid_rate": invalid / max(total, 1),
        "correct": correct,
        "invalid": invalid,
        "debug_examples": debug,
    }


def evaluate_adapter(model_key, adapter_repo):
    print("\n=== Evaluating", model_key, "===")
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    adapter_local = snapshot_download(adapter_repo, token=hf_token)
    print("Adapter snapshot:", adapter_local)

    model, tokenizer = FastVisionModel.from_pretrained(
        model_name=adapter_local,
        load_in_4bit=True,
        use_gradient_checkpointing="unsloth",
        max_seq_length=MAX_SEQ_LENGTH,
        device_map={"": 0},
    )

    FastVisionModel.for_inference(model)

    eval_sets = {
        "zahid_val": zahid_val_data,
        "zahid_test": zahid_test_data,
        "wlasl_val": wlasl_val_data,
        "wlasl_test": wlasl_test_data,
        "combined_val": combined_val_data,
        "combined_test": combined_test_data,
    }
    model_summary = {"adapter_repo": adapter_repo}
    for split_name, data in eval_sets.items():
        model_summary[split_name] = score_split(model, tokenizer, split_name, data)

    for split_name in eval_sets:
        printable = {key: value for key, value in model_summary[split_name].items() if key != "debug_examples"}
        print(model_key, split_name, printable)

    del model
    del tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return model_summary

comparison_summary = {
    "base_model": BASE_MODEL,
    "eval_datasets": {
        "zahid": {
            "zip_path": str(ZAHID_ZIP_PATH),
            "dataset_root": str(ZAHID_ROOT),
        },
        "wlasl": {
            "zip_path": str(WLASL_ZIP_PATH),
            "dataset_root": str(WLASL_ROOT),
        },
        "num_frames": NUM_FRAMES,
        "max_images_per_sample": MAX_IMAGES_PER_SAMPLE,
        "eval_max_samples": EVAL_MAX_SAMPLES,
    },
    "models": {},
    "deltas": {},
}

for model_key, adapter_repo in MODEL_REPOS.items():
    comparison_summary["models"][model_key] = evaluate_adapter(model_key, adapter_repo)

baseline_key = "notebook11_zahid_phase1"
combined_key = "notebook14_zahid_wlasl_combined"
for split_name in ["zahid_val", "zahid_test", "wlasl_val", "wlasl_test", "combined_val", "combined_test"]:
    baseline = comparison_summary["models"][baseline_key][split_name]
    combined = comparison_summary["models"][combined_key][split_name]
    comparison_summary["deltas"][split_name] = {
        "accuracy_delta_combined_minus_notebook11": combined["accuracy"] - baseline["accuracy"],
        "correct_delta_combined_minus_notebook11": combined["correct"] - baseline["correct"],
        "invalid_rate_delta_combined_minus_notebook11": combined["invalid_rate"] - baseline["invalid_rate"],
    }
    print("delta", split_name, comparison_summary["deltas"][split_name])

out_path = WORK_DIR / "model_comparison_11_vs_14.json"
out_path.write_text(json.dumps(comparison_summary, indent=2, ensure_ascii=False), encoding="utf-8")
print("Wrote comparison summary:", out_path)

print("No Hub upload performed; comparison metrics are local only.")


In [ ]:
# Visualize results
import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

assert "comparison_summary" in globals(), "Run the evaluation cell first."

VIS_DIR = WORK_DIR / "comparison_visuals"
VIS_DIR.mkdir(parents=True, exist_ok=True)

json_path = VIS_DIR / "model_comparison_11_vs_14.json"
json_path.write_text(json.dumps(comparison_summary, indent=2, ensure_ascii=False), encoding="utf-8")
print("Wrote JSON:", json_path)

rows = []
for model_key, model_result in comparison_summary["models"].items():
    for split_name, metrics in model_result.items():
        if split_name == "adapter_repo":
            continue
        rows.append({
            "model": model_key,
            "split": split_name,
            "samples": metrics["samples"],
            "accuracy": metrics["accuracy"],
            "invalid_rate": metrics["invalid_rate"],
            "correct": metrics["correct"],
            "invalid": metrics["invalid"],
        })

df = pd.DataFrame(rows)
display(df)

csv_path = VIS_DIR / "model_comparison_11_vs_14.csv"
df.to_csv(csv_path, index=False)
print("Wrote CSV:", csv_path)

pivot_acc = df.pivot(index="split", columns="model", values="accuracy")
pivot_invalid = df.pivot(index="split", columns="model", values="invalid_rate")

ax = pivot_acc.plot(kind="bar", figsize=(12, 5), ylim=(0, 1), rot=30)
ax.set_title("Notebook 11 vs Notebook 14 Accuracy")
ax.set_ylabel("Accuracy")
ax.set_xlabel("Eval split")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
acc_plot_path = VIS_DIR / "accuracy_comparison.png"
plt.savefig(acc_plot_path, dpi=160)
plt.show()
print("Wrote plot:", acc_plot_path)

ax = pivot_invalid.plot(kind="bar", figsize=(12, 5), ylim=(0, 1), rot=30)
ax.set_title("Notebook 11 vs Notebook 14 Invalid Rate")
ax.set_ylabel("Invalid rate")
ax.set_xlabel("Eval split")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
invalid_plot_path = VIS_DIR / "invalid_rate_comparison.png"
plt.savefig(invalid_plot_path, dpi=160)
plt.show()
print("Wrote plot:", invalid_plot_path)

delta_rows = []
for split_name, delta in comparison_summary["deltas"].items():
    delta_rows.append({
        "split": split_name,
        "accuracy_delta_combined_minus_notebook11": delta["accuracy_delta_combined_minus_notebook11"],
        "correct_delta_combined_minus_notebook11": delta["correct_delta_combined_minus_notebook11"],
        "invalid_rate_delta_combined_minus_notebook11": delta["invalid_rate_delta_combined_minus_notebook11"],
    })

delta_df = pd.DataFrame(delta_rows)
display(delta_df)

delta_csv_path = VIS_DIR / "model_comparison_deltas_11_vs_14.csv"
delta_df.to_csv(delta_csv_path, index=False)
print("Wrote delta CSV:", delta_csv_path)

ax = delta_df.set_index("split")["accuracy_delta_combined_minus_notebook11"].plot(kind="bar", figsize=(12, 4), rot=30)
ax.axhline(0, color="black", linewidth=1)
ax.set_title("Accuracy Delta: Notebook 14 Combined - Notebook 11 Zahid")
ax.set_ylabel("Accuracy delta")
ax.set_xlabel("Eval split")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
delta_plot_path = VIS_DIR / "accuracy_delta_combined_minus_notebook11.png"
plt.savefig(delta_plot_path, dpi=160)
plt.show()
print("Wrote plot:", delta_plot_path)
